##
$$\textbf{1 - Bloques auxiliares}$$

###
$$\textbf{Bloque 1.1 — Preparación del entorno}$$


In [10]:
import os, shutil, zipfile, subprocess, glob, re    # Librerías para gestión de archivos, carpetas y procesos del sistema
from pathlib import Path                            # Herramienta para manipular rutas de archivos de forma intuitiva
import pandas as pd                                 # Librería principal para manipulación y análisis de datos (CSVs)
import numpy as np                                  # Librería para operaciones matemáticas y manejo de arreglos numéricos
import cv2                                          # OpenCV: Librería para procesamiento de imágenes y video
import torch                                        # Framework de Deep Learning para ejecutar el modelo YOLO
from google.colab import files                      # Utilidad específica de Colab para subir y descargar archivos
import random                                       # Permite generar números aleatorios para mezclar datos antes del split
import traceback
from IPython.display import display, HTML, clear_output
from google.colab import output
import io
import contextlib
import time
import logging
import math
from skimage.morphology import skeletonize
from ultralytics import YOLO
from fpdf import FPDF
from datetime import datetime

# Instalaciones de dependencias para detección
!pip -q install ultralytics fpdf

# --- CONFIGURACIÓN DE RUTAS MAESTRAS ---
BASE = Path("/content")

# Definición de variables globales
DIR_IN = BASE/"in"
DIR_WORK = BASE/"work"
DIR_OUT = BASE/"out"
DIR_DATASET = BASE/"dataset"
DIR_RUNS = BASE/"runs"
DIR_FRAMES = DIR_WORK/"frames"
DIR_RAW = DIR_WORK/"dataset_raw"

# PATHS, para saber que carpetas debe utilizar el entorno
PATHS = {
    "in": DIR_IN,
    "work": DIR_WORK,
    "out": DIR_OUT,
    "frames": DIR_FRAMES,
    "dataset": DIR_DATASET
}

# Creación física de todo el árbol de directorios
for p in [DIR_IN, DIR_WORK, DIR_OUT, DIR_DATASET, DIR_RUNS, DIR_FRAMES, DIR_RAW]:
    p.mkdir(parents=True, exist_ok=True)

print(f"✅ Entorno preparado. GPU disponible: {torch.cuda.is_available()}")

# --- FUNCIONES DE CHEQUEO ---

def listar_carpeta(ruta, max_items=50):
    ruta = Path(ruta)
    print("\n📁", ruta)
    if not ruta.exists():
        print("⚠️ No existe todavía.")
        return
    items = sorted(list(ruta.iterdir()), key=lambda p: (p.is_file(), p.name.lower()))
    if len(items) == 0:
        print("⚠️ Está vacía.")
        return
    for p in items[:max_items]:
        tipo = "DIR " if p.is_dir() else "FILE"
        print(f" - {tipo} {p.name}")

print("\n✅ Listado rápido de carpetas del proyecto:")
listar_carpeta(BASE)
listar_carpeta(DIR_IN)

# Chequeo de espacio y peso
try:
    print("\n✅ Espacio en disco:")
    subprocess.run(["df", "-h", "/content"], check=False)
except: pass

# --- DETECCIÓN AUTOMÁTICA DE INPUTS ---
in_files = list(DIR_IN.iterdir())
video = next((p for p in in_files if p.suffix.lower() in [".mp4", ".mov", ".avi"]), None)
zip_cvat = next((p for p in in_files if p.suffix.lower() == ".zip" and "cvat" in p.name.lower()), None)
zip_images = next((p for p in in_files if p.name.lower() == "images.zip"), None)

print("\n✅ Resumen de Entradas Detectadas:")
print(" - Video:", video.name if video else "❌ No detectado")
print(" - Zip CVAT:", zip_cvat.name if zip_cvat else "❌ No detectado")
print(" - images.zip:", zip_images.name if zip_images else "❌ No detectado")

print("\n🚀 Rutas listas para el procesamiento técnico.")

✅ Entorno preparado. GPU disponible: False

✅ Listado rápido de carpetas del proyecto:

📁 /content
 - DIR  .config
 - DIR  dataset
 - DIR  in
 - DIR  out
 - DIR  runs
 - DIR  sample_data
 - DIR  work
 - FILE best.pt
 - FILE losa.csv
 - FILE paramento.csv
 - FILE Reporte_Inspeccion_Final.pdf
 - FILE TRAMO 2B.mp4
 - FILE TRAMO 2B.srt

📁 /content/in
 - FILE best.pt
 - FILE losa.csv
 - FILE paramento.csv
 - FILE TRAMO 2B.mp4
 - FILE TRAMO 2B.srt

✅ Espacio en disco:

✅ Resumen de Entradas Detectadas:
 - Video: TRAMO 2B.mp4
 - Zip CVAT: ❌ No detectado
 - images.zip: ❌ No detectado

🚀 Rutas listas para el procesamiento técnico.


###
$$\textbf{Bloque 1.2 — Subida de archivos (Video, SRT, CSVs o Modelo)}$$

In [6]:
print("📂 Selecciona archivos para añadir (Video, SRT, CSVs o Modelo)")    # Aviso de carga
subidos = files.upload()                                                   # Selector de archivos

for nombre_original in subidos.keys():                                     # Itera sobre los archivos subidos
    nombre_min = nombre_original.lower()                                   # Normaliza a minúsculas para comparar

    # --- LÓGICA DE CLASIFICACIÓN (Mantiene archivos existentes) ---
    if "losa" in nombre_min and nombre_original.endswith('.csv'):          # Identifica CSV de fondo
        ruta_destino = PATHS["in"] / "losa.csv"                            # Renombra para el motor de fusión
        print(f"🎯 Identificado como LOSA: {nombre_original}")

    elif "paramento" in nombre_min and nombre_original.endswith('.csv'):   # Identifica CSV de muros
        ruta_destino = PATHS["in"] / "paramento.csv"                       # Renombra para el motor de fusión
        print(f"🎯 Identificado como PARAMENTO: {nombre_original}")

    elif nombre_original.endswith('.pt'):                                  # Identifica modelos YOLO
        ruta_destino = PATHS["in"] / "best.pt"                             # Estandariza nombre del modelo
        print(f"🧠 Modelo cargado/actualizado: {nombre_original}")

    else:                                                                  # Video, SRT y otros
        ruta_destino = PATHS["in"] / nombre_original                       # Conserva nombre original
        print(f"📦 Archivo guardado: {nombre_original}")

    # --- GUARDADO EN DISCO ---
    with open(ruta_destino, "wb") as f:                                    # Abre destino sin borrar el resto de /in
        f.write(subidos[nombre_original])                                  # Escribe el nuevo contenido

📂 Selecciona archivos para añadir (Video, SRT, CSVs o Modelo)


Saving TRAMO 2B.srt to TRAMO 2B.srt
📦 Archivo guardado: TRAMO 2B.srt


###
$$\textbf{Bloque 1.3 — Video → Frames (FFmpeg)}$$


In [ ]:
# Busca un video por extensión común (si existe)
video = next((p for p in in_files if p.suffix.lower() in [".mp4", ".mov", ".avi", ".mkv"]), None)  # Encuentra el primer video

# Extrae frames desde el video detectado (SELECCIONAR MANUALMENTE EN CASO DE VIDEOS MÁS GRANDES)
fps_extract = 1.0                                              # Define cuántas imágenes por segundo extraer (ej: 1.0 = 1 frame/seg)
img_ext = "jpg"                                                # Define el formato de imagen de salida (jpg o png)

# Si no hay video, no hacemos nada y seguimos
if video is None:                                              # Revisa si se detectó un video en /content/in
    print("⚠️ No hay video detectado en /content/in, así que no se extraen frames todavía.")  # Aviso sin romper el flujo
else:
    # Limpia frames anteriores para no mezclar corridas
    old_frames = list(DIR_FRAMES.glob(f"*.{img_ext}"))          # Busca frames antiguos en la carpeta de frames
    for f in old_frames:                                        # Recorre frames antiguos
        f.unlink()                                              # Borra cada frame antiguo

    # Define el número inicial desde el cual deseas comenzar la numeración de imágenes
    start_number = 1

    # Define el patrón de nombres con el número de inicio
    out_pattern = str(DIR_FRAMES / f"%06d.{img_ext}")

    # Construye y ejecuta el comando ffmpeg para extraer frames
    cmd = [
    "ffmpeg", "-y",
    "-i", str(video),
    "-vf", f"fps={fps_extract}",
    "-start_number", str(start_number), # <--- ESTO indica dónde empezar
    out_pattern]

    print("✅ Ejecutando FFmpeg para extraer frames:")
    print("   ", " ".join(cmd))                                 # Muestra el comando para trazabilidad
    subprocess.run(cmd, check=False)                            # Ejecuta sin romper el notebook si ffmpeg devuelve error

    # Cuenta y muestra resultados
    frames = sorted(DIR_FRAMES.glob(f"*.{img_ext}"))            # Busca los frames generados
    if len(frames) == 0:                                        # Si no se generó ningún frame
        print("⚠️ No se generaron frames (revisa si el video está OK o si fps_extract es muy bajo).")  # Aviso
    else:
        print(f"✅ Frames listos: {len(frames)} en {DIR_FRAMES}")  # Confirma cuántos frames se crearon
        print("   Ejemplo primero/último:", frames[0].name, "|", frames[-1].name)  # Muestra nombres de ejemplo

# Comprime los frames en un ZIP descargable (si no hay frames, no falla: solo avisa)
zip_frames_path = DIR_OUT/"frames.zip"                         # Define dónde quedará el zip final

# Busca frames existentes en la carpeta de frames
frame_files = sorted(list(DIR_FRAMES.glob("*.jpg")) + list(DIR_FRAMES.glob("*.png")))  # Reúne frames jpg/png

# Si no hay frames aún, avisa y termina sin error
if len(frame_files) == 0:                                      # Revisa si hay frames para comprimir
    print("⚠️ No hay frames en /content/work/frames, así que no se puede crear frames.zip todavía.")  # Aviso
else:
    # Si ya existía un zip antiguo, lo borra para evitar confusiones
    if zip_frames_path.exists():                               # Verifica si el zip ya existe
        zip_frames_path.unlink()                               # Borra el zip anterior

    # Crea el zip con todos los frames
    with zipfile.ZipFile(zip_frames_path, "w", zipfile.ZIP_DEFLATED) as z:  # Abre un zip en modo escritura
        for f in frame_files:                                  # Recorre cada frame
            z.write(f, arcname=f.name)                         # Agrega el archivo al zip con su nombre

    # Confirma que el zip se creó correctamente
    print("✅ ZIP creado para CVAT:", zip_frames_path)          # Muestra la ruta del zip creado
    print(f"✅ Incluye {len(frame_files)} imágenes.")          # Indica cuántas imágenes quedaron dentro

    # Opción de descarga directa (si quieres)
    try:
        from google.colab import files                         # Importa herramienta de descarga de Colab
        files.download(str(zip_frames_path))                   # Descarga el zip a tu PC
        print("✅ Descarga iniciada (si tu navegador lo permite).")  # Confirmación
    except Exception:
        print("⚠️ No pude iniciar descarga automática, pero el zip quedó en /content/out para descargarlo manualmente.")  # Aviso



✅ Ejecutando FFmpeg para extraer frames:
    ffmpeg -y -i /content/in/TRAMO 3 - PARTE 2.mp4 -vf fps=1.0 -start_number 590 /content/work/frames/%06d.jpg
✅ Frames listos: 296 en /content/work/frames
   Ejemplo primero/último: 000590.jpg | 000885.jpg
✅ ZIP creado para CVAT: /content/out/frames.zip
✅ Incluye 296 imágenes.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga iniciada (si tu navegador lo permite).


##
$$\textbf{2 - Entrenamiento del modelo}$$

###
$$\textbf{Bloque 2.1 — Importar zip CVAT y normalizar train/val para su uso, + definir clases}$$



In [ ]:
# Descomprime el export de CVAT en /content/work/dataset_raw
# Nota: Este zip debe llamarse de forma que contenga "cvat" en el nombre

# Vuelve a buscar un zip que parezca de CVAT
in_files = list(DIR_IN.iterdir())                                              # Lee archivos en /content/in
zip_cvat = next((p for p in in_files if p.suffix.lower() == ".zip" and "cvat" in p.name.lower()), None)  # Busca zip con "cvat" en el nombre

if zip_cvat is None:                                                           # Revisa si existe export de CVAT
    print("⚠️ No detecté ningún .zip de CVAT en /content/in (debe tener 'cvat' en el nombre).")           # Aviso
    print("ℹ️ Cuando lo tengas, súbelo y vuelve a correr este bloque.")         # Guía
else:
    if DIR_RAW.exists():                                                       # Revisa si ya existe dataset_raw
        shutil.rmtree(DIR_RAW)                                                 # Borra dataset_raw anterior completo
    DIR_RAW.mkdir(parents=True, exist_ok=True)                                 # Crea dataset_raw limpio

    # Descomprime el zip de CVAT dentro de dataset_raw
    with zipfile.ZipFile(zip_cvat, "r") as z:                                  # Abre el zip de CVAT
        z.extractall(DIR_RAW)                                                  # Extrae todo su contenido en DIR_RAW

    # Lista contenido para confirmar que se extrajo algo
    extracted_any = any(DIR_RAW.rglob("*"))                                     # Verifica si hay archivos extraídos
    if not extracted_any:                                                      # Si no se extrajo nada
        print("⚠️ El zip se descomprimió, pero no veo archivos dentro. Revisa si el zip está correcto.")  # Aviso
    else:
        print("✅ Export CVAT descomprimido en:", DIR_RAW)                      # Confirmación
        # Muestra un vistazo rápido de archivos/carpetas extraídas
        top_items = sorted(list(DIR_RAW.iterdir()))                             # Lista el primer nivel de dataset_raw
        print("✅ Primer nivel dentro de dataset_raw:")                         # Título del listado
        for p in top_items[:30]:                                                # Muestra hasta 30 ítems
            tag = "DIR " if p.is_dir() else "FILE"                              # Marca si es carpeta o archivo
            print(" -", tag, p.name)                                            # Imprime nombre
        if len(top_items) > 30:                                                 # Si hay muchos ítems
            print(f" ... y {len(top_items)-30} más")                            # Indica que hay más


# Detecta imágenes/labels en dataset_raw y construye dataset YOLO final (train/val) creando .txt vacíos para negativos
img_exts = {".jpg", ".jpeg", ".png"}                             # Extensiones válidas de imágenes
train_ratio = 0.8                                                # Proporción train/val
seed = 42                                                        # Semilla para split repetible

# Define rutas YOLO estándar de salida
IMG_TRAIN = DIR_DATASET/"images/train"                           # Imágenes train
IMG_VAL   = DIR_DATASET/"images/val"                             # Imágenes val
LBL_TRAIN = DIR_DATASET/"labels/train"                           # Labels train
LBL_VAL   = DIR_DATASET/"labels/val"                             # Labels val

# Crea carpetas de salida
for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:               # Recorre carpetas YOLO
    d.mkdir(parents=True, exist_ok=True)                         # Crea si falta

# Verifica que exista dataset_raw
if not DIR_RAW.exists():                                         # Si no existe dataset_raw
    print("⚠️ No existe /content/work/dataset_raw todavía. Corre el Bloque 8 (descomprimir CVAT) primero.")  # Aviso
else:
    # Busca imágenes y txt dentro de dataset_raw (recursivo)
    all_imgs = [p for p in DIR_RAW.rglob("*") if p.suffix.lower() in img_exts]   # Todas las imágenes
    all_txt  = [p for p in DIR_RAW.rglob("*.txt")]                               # Todos los txt

    # Si falta algo, avisa pero no rompe
    if len(all_imgs) == 0:
        print("⚠️ No encontré imágenes dentro de dataset_raw. Revisa estructura del export (o tu zip).")  # Aviso
    if len(all_txt) == 0:
        print("⚠️ No encontré labels (.txt) dentro de dataset_raw. Ojo: CVAT solo exporta txt con anotaciones.")  # Aviso

    # Detecta carpeta con más imágenes y carpeta con más txt (candidatas principales)
    img_dir = None                                               # Carpeta candidata de imágenes
    lbl_dir = None                                               # Carpeta candidata de labels

    if len(all_imgs) > 0:                                        # Si hay imágenes
        counts_img_parent = {}                                   # Conteo por carpeta
        for p in all_imgs:                                       # Recorre imágenes
            counts_img_parent[p.parent] = counts_img_parent.get(p.parent, 0) + 1  # Cuenta
        img_dir = max(counts_img_parent, key=counts_img_parent.get)              # Elige mayor

    if len(all_txt) > 0:                                         # Si hay txt
        counts_lbl_parent = {}                                   # Conteo por carpeta
        for p in all_txt:                                        # Recorre txt
            counts_lbl_parent[p.parent] = counts_lbl_parent.get(p.parent, 0) + 1  # Cuenta
        lbl_dir = max(counts_lbl_parent, key=counts_lbl_parent.get)              # Elige mayor

    print("\n✅ Rutas detectadas (candidatas principales):")      # Imprime rutas
    print(" - img_dir:", str(img_dir) if img_dir else "No detectado")  # Carpeta imágenes
    print(" - lbl_dir:", str(lbl_dir) if lbl_dir else "No detectado")  # Carpeta labels

    # Si no se detectaron rutas, termina sin error
    if (img_dir is None) or (lbl_dir is None):
        print("⚠️ No pude detectar img_dir y/o lbl_dir. Revisa qué hay dentro de dataset_raw (Bloque 4 listar carpetas).")  # Aviso
    else:
        # Crea mapas por stem para hacer match imagen <-> label
        img_map = {}                                             # stem -> ruta imagen
        for p in img_dir.iterdir():                              # Recorre archivos en img_dir
            if p.suffix.lower() in img_exts:                     # Si es imagen
                img_map[p.stem] = p                              # Guarda

        lbl_map = {}                                             # stem -> ruta label
        for p in lbl_dir.iterdir():                              # Recorre archivos en lbl_dir
            if p.suffix.lower() == ".txt":                       # Si es txt
                lbl_map[p.stem] = p                              # Guarda

        # Diagnóstico de matching
        all_stems = sorted(img_map.keys())                       # Todas las imágenes (incluye negativos)
        paired = sorted(set(img_map.keys()) & set(lbl_map.keys()))            # Con label
        missing_lbl = sorted(set(img_map.keys()) - set(lbl_map.keys()))      # Sin label (negativos)
        missing_img = sorted(set(lbl_map.keys()) - set(img_map.keys()))      # Txt sin imagen

        print("\n✅ Matching imagen + label (incluyendo negativos):")  # Reporte matching
        print(" - Total imágenes:", len(all_stems))              # Total
        print(" - Con label (.txt):", len(paired))               # Con txt
        print(" - Sin label (se creará .txt vacío):", len(missing_lbl))  # Negativos

        # Si no hay imágenes, no se puede construir dataset
        if len(all_stems) == 0:
            print("⚠️ No hay imágenes válidas para construir dataset. Revisa nombres/extensiones.")  # Aviso
        else:
            # Limpia salidas anteriores
            for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:   # Recorre carpetas de salida
                for f in d.glob("*"):                            # Recorre archivos dentro
                    f.unlink()                                   # Borra

# ------------------------------------------------------------------------------------------------------------------------------------

# Split train/val sobre TODAS las imágenes

            random.seed(seed)                                    # Semilla
            random.shuffle(all_stems)                             # Mezcla
            cut = int(len(all_stems) * train_ratio)              # Corte
            train_ids = all_stems[:cut]                          # Train stems
            val_ids = all_stems[cut:]                            # Val stems

            # Copia imagen y label (o crea vacío si falta)
            def copy_img_and_label(stem, img_dst, lbl_dst):      # Función copiar + label
                img_src = img_map[stem]                          # Imagen fuente
                shutil.copy2(img_src, img_dst / img_src.name)    # Copia imagen

                lbl_out = lbl_dst / f"{stem}.txt"                # Label destino
                if stem in lbl_map:                              # Si existe label real
                    shutil.copy2(lbl_map[stem], lbl_out)         # Copia label
                else:
                    lbl_out.write_text("", encoding="utf-8")     # Crea label vacío (negativo)

            # Copia a train
            for s in train_ids:                                  # Recorre train
                copy_img_and_label(s, IMG_TRAIN, LBL_TRAIN)      # Copia

            # Copia a val
            for s in val_ids:                                    # Recorre val
                copy_img_and_label(s, IMG_VAL, LBL_VAL)          # Copia

            # Resumen final
            n_train_img = len(list(IMG_TRAIN.glob("*")))         # Cuenta imgs train
            n_train_lbl = len(list(LBL_TRAIN.glob("*.txt")))     # Cuenta labels train
            n_val_img = len(list(IMG_VAL.glob("*")))             # Cuenta imgs val
            n_val_lbl = len(list(LBL_VAL.glob("*.txt")))         # Cuenta labels val

            print("\n✅ Dataset YOLO final creado en:", DIR_DATASET)  # Confirma creación
            print(" - Train: imgs =", n_train_img, "| lbls =", n_train_lbl)  # Resumen train
            print(" - Val:   imgs =", n_val_img,   "| lbls =", n_val_lbl)    # Resumen val

            # Chequeo 1 txt por imagen (ideal)
            if n_train_img != n_train_lbl:
                print("⚠️ Ojo: TRAIN imgs != txt (deberían ser iguales).")  # Aviso
            if n_val_img != n_val_lbl:
                print("⚠️ Ojo: VAL imgs != txt (deberían ser iguales).")    # Aviso

# ------------------------------------------------------------------------------------------------------------------------------------

# Crea el archivo data.yaml para YOLO y DEFINE MANUALMENTE las clases

yaml_path = BASE/"data.yaml"                                    # Define la ruta donde se guardará el YAML

# Define aquí tus clases EXACTAS y en el MISMO orden que usaste en CVAT - OJO DEBE HACERSE MANUAL
classes = [
    "daño junta paramento izquierdo",
    "daño junta paramento derecho",
    "daño junta losa fondo",
    "estría lado izquierdo",
    "estría lado derecho",
    "estría centro",
    "daño paramento",
    "daño losa fondo",
]

# Verifica que existan carpetas train/val para evitar un YAML apuntando a nada
train_dir_ok = (DIR_DATASET/"images/train").exists()            # Revisa si existe la carpeta de imágenes train
val_dir_ok = (DIR_DATASET/"images/val").exists()                # Revisa si existe la carpeta de imágenes val

# Si no hay estructura dataset, avisa y no rompe el notebook
if not (train_dir_ok and val_dir_ok):                           # Si faltan carpetas básicas del dataset
    print("⚠️ No detecto la estructura de dataset en /content/dataset/images/train y /val.")  # Aviso
    print("ℹ️ Corre el Bloque 10 (normalización + split) antes de crear el data.yaml.")       # Guía
else:
    # Si no definiste clases, avisa para que no entrenes con un YAML incompleto
    if len(classes) == 0:                                       # Si la lista de clases está vacía
        print("⚠️ La lista 'classes' está vacía. Agrega tus clases en el orden de CVAT antes de entrenar.")  # Aviso
        print("ℹ️ Igual crearé el data.yaml, pero NO deberías entrenar hasta completar 'classes'.")          # Guía

    # Construye el contenido del YAML que YOLO necesita
    yaml_text = f"""path: {DIR_DATASET}
train: images/train
val: images/val
names:
"""                                                             # Texto base del YAML (path + rutas train/val)

    # Agrega las clases con su índice (0,1,2...) en el orden correcto
    for i, c in enumerate(classes):                              # Recorre clases con índice
        yaml_text += f"  {i}: {c}\n"                              # Agrega cada clase al YAML

    # Guarda el archivo data.yaml
    yaml_path.write_text(yaml_text, encoding="utf-8")            # Escribe el YAML en disco

    # Prints de confirmación + vista rápida del contenido
    print("✅ data.yaml creado en:", yaml_path)                   # Confirma ruta del archivo creado
    print("✅ Contenido de data.yaml:")                           # Título del contenido
    print(yaml_text)                                              # Muestra el texto completo

⚠️ No detecté ningún .zip de CVAT en /content/in (debe tener 'cvat' en el nombre).
ℹ️ Cuando lo tengas, súbelo y vuelve a correr este bloque.
⚠️ No encontré imágenes dentro de dataset_raw. Revisa estructura del export (o tu zip).
⚠️ No encontré labels (.txt) dentro de dataset_raw. Ojo: CVAT solo exporta txt con anotaciones.

✅ Rutas detectadas (candidatas principales):
 - img_dir: No detectado
 - lbl_dir: No detectado
⚠️ No pude detectar img_dir y/o lbl_dir. Revisa qué hay dentro de dataset_raw (Bloque 4 listar carpetas).
✅ data.yaml creado en: /content/data.yaml
✅ Contenido de data.yaml:
path: /content/dataset
train: images/train
val: images/val
names:
  0: daño junta paramento izquierdo
  1: daño junta paramento derecho
  2: daño junta losa fondo
  3: estría lado izquierdo
  4: estría lado derecho
  5: estría centro
  6: daño paramento
  7: daño losa fondo



###
$$\textbf{Bloque 2.2 — Entrena YOLO usando dataset etiquetado, guardando best.pt como mejor versión del modelo}$$


In [ ]:
# 1. Configuración de parámetros
data_yaml = "/content/data.yaml"
batch_size = 16
epochs = 215
img_size = 640
output_dir = "/content/runs"
best_pt_path = Path("/content/in/best.pt")

# Asegurar que la carpeta /content/in existe
best_pt_path.parent.mkdir(parents=True, exist_ok=True)

# 2. Lógica de carga de modelo (Continuar Partida o Empezar de Cero)
if best_pt_path.exists():
    print(f"🎮 'Save Game' detectado. Cargando progreso desde: {best_pt_path}")
    model = YOLO(str(best_pt_path))
    lr_inicial = 0.001 # Tasa más baja para no arruinar lo ya aprendido
else:
    print("🆕 No hay partida guardada. Iniciando entrenamiento desde cero (YOLOv8m).")
    model = YOLO("yolov8l.pt")
    lr_inicial = 0.01

# 3. Entrenamiento
print(f"🚀 Iniciando entrenamiento (Máximo {epochs} epochs)...")
results = model.train(
    data=data_yaml,
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    project=output_dir,
    name="train_experiment",
    exist_ok=True,
    lr0=lr_inicial,
    patience=30,      # Si deja de mejorar por 30 epochs, se detiene
    save=True,
    pretrained=True
)

# 4. Gestión de archivos y Auto-Descarga ("Guardado Final")
print("✅ Entrenamiento finalizado.")

# Ruta donde YOLO acaba de guardar el mejor modelo de esta sesión
new_best_path = Path(output_dir) / "train_experiment" / "weights" / "best.pt"

if new_best_path.exists():
    # 4a. Sobreescribir el Save Game en /content/in (Para la próxima vez que corras el bloque)
    shutil.copy2(new_best_path, best_pt_path)
    print(f"⭐ 'Save Game' actualizado localmente en: {best_pt_path}")

    # 4b. Descargar el archivo al PC automáticamente
    try:
        print("📥 Iniciando descarga del modelo 'best.pt' a tu ordenador...")
        files.download(str(best_pt_path))
        print("✅ Descarga iniciada. ¡Guarda bien este archivo!")
    except Exception as e:
        print(f"⚠️ No se pudo iniciar la descarga automática: {e}")
        print("ℹ️ Puedes descargarlo manualmente desde la carpeta /content/in en el panel izquierdo.")
else:
    print("⚠️ Error: No se encontró el archivo generado en esta sesión.")

🎮 'Save Game' detectado. Cargando progreso desde: /content/in/best.pt
🚀 Iniciando entrenamiento (Máximo 215 epochs)...
Ultralytics 8.4.15 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=215, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/in/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

###
$$\textbf{Bloque 2.3 — Usa best.pt para predecir sobre images/val y guarda imágenes con cajas dibujadas}$$


In [ ]:
# --- 1. CONFIGURACIÓN DE RUTAS ---
base_path = Path("/content")
dir_dataset = base_path / "dataset"
dir_runs = base_path / "runs"
dir_out = base_path / "out"
dir_in = base_path / "in"

# Ruta del modelo y de las imágenes
best_model_path = dir_in / "best.pt"
val_images_path = dir_dataset / "images/val"

# Carpeta de salida
pred_name = "predict_val"
final_pred_dir = dir_runs / pred_name

# --- 2. VALIDACIÓN ---
if not best_model_path.exists():
    print(f"⚠️ No se encuentra el modelo en {best_model_path}. Revisa el Bloque 12.")
elif not val_images_path.exists():
    print(f"⚠️ No existe la carpeta de validación en {val_images_path}")
else:
    try:
        print(f"🚀 Iniciando predicción visual con líneas delgadas...")
        model = YOLO(str(best_model_path))

        # Ejecutamos la predicción con ajustes visuales
        model.predict(
            source=str(val_images_path),
            save=True,
            project=str(dir_runs),
            name=pred_name,
            exist_ok=True,
            conf=0.25,         # Solo muestra lo que tenga > 25% certeza
            iou=0.3,          # Umbral de solapamiento (ajustar si hay cajas duplicadas)
            # --- AJUSTES PARA QUE NO SE TAPEN LAS FALLAS ---
            line_width=3,
            show_labels=True,  # Muestra el nombre de la falla
            show_conf=False,   # Quitamos el % de confianza para limpiar la imagen
            save_txt=False     # No necesitamos los .txt aquí, solo las fotos
        )

        print(f"✅ Fotos guardadas en: {final_pred_dir}")

        # --- 3. CREACIÓN DE ZIP Y DESCARGA ---
        zip_file_path = dir_out / "val_predictions_clean.zip"
        dir_out.mkdir(parents=True, exist_ok=True)

        # Empaquetamos solo las imágenes resultantes
        with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # Buscamos extensiones comunes de imagen
            for ext in ['*.jpg', '*.jpeg', '*.png']:
                for img_file in final_pred_dir.rglob(ext):
                    zipf.write(img_file, arcname=img_file.name)

        if zip_file_path.stat().st_size > 0:
            print(f"📦 ZIP creado: {zip_file_path}")
            files.download(str(zip_file_path))
            print("📥 Descarga iniciada.")
        else:
            print("⚠️ El ZIP está vacío. ¿Seguro que hubo detecciones?")

    except Exception as e:
        print(f"❌ Error: {e}")

🚀 Iniciando predicción visual con líneas delgadas...

image 1/253 /content/dataset/images/val/000002.jpg: 384x640 (no detections), 54.0ms
image 2/253 /content/dataset/images/val/000003.jpg: 384x640 1 daño paramento, 31.0ms
image 3/253 /content/dataset/images/val/000007.jpg: 384x640 (no detections), 31.0ms
image 4/253 /content/dataset/images/val/000014.jpg: 384x640 2 estría centros, 31.0ms
image 5/253 /content/dataset/images/val/000015.jpg: 384x640 2 estría centros, 31.0ms
image 6/253 /content/dataset/images/val/000024.jpg: 384x640 2 estría centros, 31.0ms
image 7/253 /content/dataset/images/val/000031.jpg: 384x640 2 estría centros, 28.8ms
image 8/253 /content/dataset/images/val/000040.jpg: 384x640 2 estría lado derechos, 1 estría centro, 25.2ms
image 9/253 /content/dataset/images/val/000044.jpg: 384x640 2 estría lado derechos, 24.9ms
image 10/253 /content/dataset/images/val/000052.jpg: 384x640 2 estría lado derechos, 25.2ms
image 11/253 /content/dataset/images/val/000055.jpg: 384x640 2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descarga iniciada.


##
$$\textbf{3 - Ejecución del Modelo DAICH}$$

In [17]:
# --- VALIDACIÓN PREVIA DE ARCHIVOS CRÍTICOS ---
archivos_faltantes = []

# 1. Verificar Video (.mp4)
video_check = list(PATHS["in"].glob("*.mp4"))
if not video_check: archivos_faltantes.append("Video (.mp4)")
else: print(f"✅ Video detectado: {video_check[0].name}")

# 2. Verificar Modelo (best.pt)
modelo_path = PATHS["in"] / "best.pt"
if not modelo_path.exists(): archivos_faltantes.append("Modelo (best.pt)")
else: print(f"✅ Modelo detectado: best.pt")

# 3. Verificar Telemetría (.srt)
srt_check = list(PATHS["in"].glob("*.srt"))
if not srt_check: archivos_faltantes.append("Telemetría (.srt)")
else: print(f"✅ SRT detectado: {srt_check[0].name}")

# 4. Verificar CSV Losa
losa_path = PATHS["in"] / "losa.csv"
if not losa_path.exists(): archivos_faltantes.append("CSV de Losa (losa.csv)")
else: print(f"✅ Datos de Losa detectados")

# 5. Verificar CSV Paramento
paramento_path = PATHS["in"] / "paramento.csv"
if not paramento_path.exists(): archivos_faltantes.append("CSV de Paramento (paramento.csv)")
else: print(f"✅ Datos de Paramento detectados")

# --- CONTROL DE DETENCIÓN
if archivos_faltantes:
    print("\n❌ ERROR: Faltan los siguientes archivos en la carpeta de entrada:")
    for arch in archivos_faltantes:
        print(f"   - {arch}")
    raise SystemExit("Deteniendo ejecución: Faltan requisitos para procesar el reporte.")
else:
    print("\n🚀 Todos los componentes listos. Iniciando procesamiento...\n" + "-"*50)

# ----------------------------------------------------------------------------------------------------------------------------------

# Analiza los archivos SRT para datos georeferenciales y CSVs para datos técnicos

def procesar_telemetria_srt(ruta_srt):
    print(f"📖 Leyendo archivo: {ruta_srt.name}")
    with open(ruta_srt, 'r', encoding='utf-8-sig', errors='ignore') as f:
        contenido = f.read()

    patron_tiempo = r'(\d{1,2}:\d{1,2}:\d{1,2}(?:[.,]\d+)?) --> (\d{1,2}:\d{1,2}:\d{1,2}(?:[.,]\d+)?)'
    tiempos = re.findall(patron_tiempo, contenido)
    partes = re.split(patron_tiempo, contenido)
    textos = partes[3::3]

    mapeo = []
    for i, (inicio, fin) in enumerate(tiempos):
        try:
            partes_hms = inicio.replace(',', '.').split(':')
            seg_total = int(partes_hms[0]) * 3600 + int(partes_hms[1]) * 60 + float(partes_hms[2])
            texto_bloque = textos[i] if i < len(textos) else ""
            texto_limpio = re.sub(r'<.*?>', '', texto_bloque).strip()
            numeros = re.findall(r"[-+]?\d*\.\d+|\d+", texto_limpio)
            if numeros:
                mapeo.append({'s': seg_total, 'm': float(numeros[-1])})
        except: continue
    return pd.DataFrame(mapeo)

print(f"✅ SRT leído correctamente")

# --- CONFIGURACIÓN DE UMBRAL DE DATOS TÉCNICOS ---
UMBRAL_DISTANCIA = 3.0

def cargar_datos_robot(ruta):
    try:
        df = pd.read_csv(ruta, sep=';', decimal=',', encoding='latin-1')
    except:
        df = pd.read_csv(ruta, sep=';', decimal=',', encoding='cp1252')
    return df.loc[:, ~df.columns.str.contains('^Unnamed')]

print("🧠 Fusionando datos con regla de proximidad estricta (+3m)...")
df_l = cargar_datos_robot(PATHS["in"] / "losa.csv")
df_p = cargar_datos_robot(PATHS["in"] / "paramento.csv")

datos_finales = []

# ----------------------------------------------------------------------------------------------------------------------------------

# Analiza el video usando el Modelo DAICH y se ubica espacialmente usando el SRT

model = YOLO(PATHS["in"] / "best.pt")
video_path = next(PATHS["in"].glob("*.mp4"))
df_telemetria = procesar_telemetria_srt(next(PATHS["in"].glob("*.srt")))

if df_telemetria.empty:
    print("❌ ABORTANDO: El archivo SRT no tiene el formato correcto.")
else:
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duracion_video = total_frames / fps
    hallazgos_crudos = []
    f_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if f_count % int(fps) == 0:
            seg_act = f_count / fps
            m_act = np.interp(seg_act, df_telemetria['s'], df_telemetria['m'])
            preds = model.predict(frame, conf=0.25, verbose=False)
            for r in preds:
                if len(r.boxes) > 0:
                    frame_plot = r.plot(conf=False, line_width=2, font_size=0.6)
                    # RECORTE ESPECÍFICO (Filtra franjas negras: de 83 a 257)
                    frame_rec = frame_plot[83:257, :]

                    img_f = f"evidencia_m_{m_act:.2f}.jpg"
                    cv2.imwrite(str(PATHS["frames"] / img_f), frame_rec)

                    clases_detectadas = list(set([model.names[int(b.cls)] for b in r.boxes]))
                    hallazgos_crudos.append({
                        'metro': round(m_act, 1),
                        'seg_video': seg_act,
                        'clases': clases_detectadas,
                        'n_clases': len(clases_detectadas),
                        'img': img_f
                    })
            print(f"🔍 Escaneando Metro: {m_act:.2f}", end="\r")
        f_count += 1
    cap.release()
    df_vis = pd.DataFrame(hallazgos_crudos).drop_duplicates(subset=['metro'])
    print(f"\n✅ Análisis visual terminado.")

# ----------------------------------------------------------------------------------------------------------------------------------

# Relaciona las detecciones del video con los datos técnicos disponibles (CSVs)

for _, det in df_vis.iterrows():
    clase_ia = det['clases'][0].lower()
    db, origen = (df_l, "losa.csv") if any(k in clase_ia for k in ["estría", "losa", "fondo"]) else (df_p, "paramento.csv")
    db['diff_lineal'] = db['Metros'] - det['metro']
    validos = db[(db['diff_lineal'] >= 0) & (db['diff_lineal'] <= UMBRAL_DISTANCIA)]

    res = {**det, 'csv_usado': origen}
    res['nombre_clase'] = "Múltiples clases detectadas" if det['n_clases'] > 1 else det['clases'][0].upper()

    if not validos.empty:
        cercano = validos.nsmallest(1, 'diff_lineal').iloc[0]
        res.update({'tiene_datos': True, 'Magnitud': cercano.get('Magnitud', 'N/A'),
                    'Extensión (cm)': cercano.get('Extensión (cm)', 0), 'Alto (cm)': cercano.get('Alto (cm)', 0),
                    'Área (m2)': cercano.get('Área (m2)', 0), 'Volumen (m3)': cercano.get('Volumen (m3)', 0),
                    'Profundidad (cm)': cercano.get('Profundidad (cm)', 0)})
    else:
        res.update({'tiene_datos': False, 'Profundidad (cm)': -1, 'Magnitud': 'N/A'})
    datos_finales.append(res)

df_reporte = pd.DataFrame(datos_finales)

# ----------------------------------------------------------------------------------------------------------------------------------

# Construye el documento PDF con las evidencias capturadas y los datos técnicos

class PDF(FPDF):
    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(120)
        self.cell(0, 10, 'Gerencia de Tranques, Relaves y Recursos Hídricos (GTRH) - CODELCO', 0, 0, 'C')
        self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'R')

pdf = PDF()
pdf.set_auto_page_break(auto=True, margin=20)

def fmt_km(val): return f"{val:,.1f}".replace(",", ".")
def fmt_t(s): return f"{int(s//60)}m {int(s%60)}s"

# --- PASO 1: PORTADA ---
pdf.add_page()
pdf.ln(50)
pdf.set_font("Arial", 'B', 22)
pdf.cell(190, 15, "INFORME TÉCNICO DE INSPECCIÓN", ln=True, align='C')
pdf.set_font("Arial", 'B', 14)
pdf.cell(190, 10, "SISTEMA DE VISIÓN ARTIFICIAL DAICH", ln=True, align='C')
pdf.ln(10)
pdf.set_font("Arial", '', 11)
pdf.cell(190, 8, f"Modelo Utilizado: DAICH YOLOv8", ln=True, align='C')
pdf.cell(190, 8, f"Fecha: {datetime.now().strftime('%d/%m/%Y')}", ln=True, align='C')
pdf.ln(30)
pdf.cell(190, 10, "(Espacio para Logo CODELCO / GTRH)", ln=True, align='C')

# --- PASO 2: CUERPO (Generamos los hallazgos para saber sus páginas) ---
mapa_paginas = {}
for idx, f in df_reporte.iterrows():
    pdf.add_page()
    mapa_paginas[idx] = pdf.page_no()

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, f"HALLAZGO: {f['nombre_clase']}", ln=True, border='B')
    pdf.ln(4)

    if f['tiene_datos']:
        pdf.set_font("Arial", 'B', 9); pdf.set_fill_color(245, 245, 245)
        # Fila 1
        pdf.cell(47.5, 8, " Kilometraje (KM)", 1, 0, 'L', True); pdf.cell(47.5, 8, fmt_km(f['metro']), 1, 0, 'C')
        pdf.cell(47.5, 8, " Tiempo Video", 1, 0, 'L', True); pdf.cell(47.5, 8, fmt_t(f['seg_video']), 1, 1, 'C')
        # Fila 2
        pdf.cell(47.5, 8, " Extensión (cm)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Extensión (cm)']:.2f}", 1, 0, 'C')
        pdf.cell(47.5, 8, " Ancho (cm)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Alto (cm)']:.2f}", 1, 1, 'C')
        # Fila 3
        pdf.cell(47.5, 8, " Área (m2)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Área (m2)']:.4f}", 1, 0, 'C')
        pdf.cell(47.5, 8, " Volumen (m3)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Volumen (m3)']:.4f}", 1, 1, 'C')
        # Fila 4
        pdf.cell(47.5, 8, " Profundidad (cm)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Profundidad (cm)']:.2f}", 1, 0, 'C')
        pdf.cell(47.5, 8, " Magnitud", 1, 0, 'L', True); pdf.cell(47.5, 8, str(f['Magnitud']), 1, 1, 'C')
    else:
        pdf.set_text_color(200, 0, 0)
        pdf.multi_cell(190, 10, f"KM {fmt_km(f['metro'])} | AVISO: SIN COINCIDENCIA TÉCNICA.", border=1, align='C')
        pdf.set_text_color(0, 0, 0)

    pdf.ln(5)
    pdf.image(str(PATHS["frames"]/f['img']), x=10, w=190)
    pdf.ln(2)
    pdf.set_font("Arial", 'I', 7)
    pdf.cell(190, 5, "Imagen extraída de registro de inspección realizado por Maquintel", ln=True, align='C')

# --- PASO 3: RESUMEN (Ahora con las páginas reales) ---
pdf.add_page()
inicio_resumen = pdf.page_no()
pdf.set_font("Arial", 'B', 14)
pdf.cell(190, 10, "RESUMEN TÉCNICO DE DETECCIONES", ln=True, align='C')
pdf.ln(5)
pdf.set_font("Arial", 'B', 8); pdf.set_fill_color(230, 230, 230)
headers = ["Tiempo", "KM", "Clase de Daño", "Magnitud", "Profundidad", "Pág"]
w_cols = [20, 25, 75, 20, 25, 10]
for h, w in zip(headers, w_cols): pdf.cell(w, 8, h, 1, 0, 'C', True)
pdf.ln()

df_res = df_reporte[df_reporte['tiene_datos']].sort_values(by='Profundidad (cm)', ascending=False)
pdf.set_font("Arial", '', 8)

for idx, r in df_res.iterrows():
    # Estimamos cuántas páginas ocupa el resumen para ajustar el índice
    num_pags_resumen = pdf.page_no() - inicio_resumen + 1
    pdf.cell(20, 7, fmt_t(r['seg_video']), 1, 0, 'C')
    pdf.cell(25, 7, fmt_km(r['metro']), 1, 0, 'C')
    pdf.cell(75, 7, r['nombre_clase'][:45], 1, 0, 'L')
    pdf.cell(20, 7, str(r['Magnitud']), 1, 0, 'C')
    pdf.cell(25, 7, f"{r['Profundidad (cm)']:.2f}", 1, 0, 'C')
    # Sumamos el desplazamiento que causó insertar el resumen al principio
    pdf.cell(10, 7, str(mapa_paginas[idx] + num_pags_resumen), 1, 1, 'C')

# --- PASO 4: REORDENAR (Mover el resumen a la posición 2) ---
total_actual = len(pdf.pages)
paginas_del_resumen = total_actual - inicio_resumen + 1

# Extraemos las páginas del resumen del final y las ponemos al principio
for i in range(paginas_del_resumen):
    # Usamos len(pdf.pages)-1 para sacar siempre la última actual
    resumen_content = pdf.pages.pop(len(pdf.pages) - 1)
    pdf.pages.insert(1, resumen_content)

print("📄 Reporte finalizado.")
report_name = "Reporte_GTRH_Final.pdf"
pdf.output(report_name)

✅ Video detectado: TRAMO 2B.mp4
✅ Modelo detectado: best.pt
✅ SRT detectado: TRAMO 2B.srt
✅ Datos de Losa detectados
✅ Datos de Paramento detectados

🚀 Todos los componentes listos. Iniciando procesamiento...
--------------------------------------------------
✅ SRT leído correctamente
🧠 Fusionando datos con regla de proximidad estricta (+3m)...
📖 Leyendo archivo: TRAMO 2B.srt


KeyboardInterrupt: 

##
$$\textbf{4 — Bloque prototipo de auto-detección de dimensiones}$$


In [ ]:
# --- RUTAS ---
DIR_IN = '/content/in'
DIR_OUT = '/content/out/Analisis_ECM_Final'

if os.path.exists(DIR_OUT):
    shutil.rmtree(DIR_OUT)
os.makedirs(DIR_OUT, exist_ok=True)

# --- ESCALA ---
ANCHO_INICIAL = 1000  # mm

# --- FACTORES DE CORRECCIÓN AJUSTADOS PARA CASO ESTUDIO ---
FACTOR_LONGITUD = 1.69
FACTOR_ANCHO = 8.79

# --- RANGOS DEFINIDOS PARA CASO ESTUDIO
MIN_LARGO = 10    # cm
MAX_LARGO = 490  # cm

MIN_ANCHO = 4   # cm
MAX_ANCHO = 26     # cm


def calcular_longitud_skeleton(skeleton):
    sk = skeleton.astype("uint8")
    coords = list(zip(*sk.nonzero()))
    length = 0.0

    for y, x in coords:
        for dy in [-1,0,1]:
            for dx in [-1,0,1]:

                if dx == 0 and dy == 0:
                    continue

                ny = y + dy
                nx = x + dx

                if ny < 0 or nx < 0 or ny >= sk.shape[0] or nx >= sk.shape[1]:
                    continue

                if sk[ny, nx]:
                    if abs(dx)==1 and abs(dy)==1:
                        length += math.sqrt(2)
                    else:
                        length += 1

    return length / 2


def procesar_fijo_final():

    zip_path = next((f for f in os.listdir(DIR_IN) if f.endswith('.zip')), None)
    if not zip_path:
        return print("❌ Sube el ZIP a /content/in")

    tmp = '/content/tmp'
    if os.path.exists(tmp):
        shutil.rmtree(tmp)

    shutil.unpack_archive(os.path.join(DIR_IN, zip_path), tmp)

    img_dir = next((r for r,d,f in os.walk(tmp) if 'images' in r and f), None)
    lbl_dir = next((r for r,d,f in os.walk(tmp) if 'labels' in r and f), None)

    if not img_dir or not lbl_dir:
        return print("❌ Estructura inválida")

    print("🚀 Procesando...")

    for txt_file in os.listdir(lbl_dir):

        if not txt_file.endswith(".txt"):
            continue

        img_name = txt_file.replace(".txt",".jpg")
        path_img = os.path.join(img_dir,img_name)

        if not os.path.exists(path_img):
            continue

        img_full = cv2.imread(path_img)
        if img_full is None:
            continue

        h_img, w_img = img_full.shape[:2]
        mitad = w_img // 2

        # --- MITAD DERECHA ---
        img = img_full[:, mitad:].copy()
        canvas = img.copy()

        px_to_mm = ANCHO_INICIAL / float(mitad)

        with open(os.path.join(lbl_dir,txt_file)) as f:
            lines = [l.strip() for l in f if l.strip()]

        hay_datos = False

        for line in lines:

            parts = line.split()
            if len(parts) < 5:
                continue

            try:
                vals = [float(x) for x in parts]
            except:
                continue

            coords = vals[1:]

            # --- BBOX ---
            if len(coords)==4:
                xc,yc,wb,hb = coords

                ancho = wb*w_img
                alto = hb*h_img

                x1 = int((xc*w_img)-ancho/2)
                y1 = int((yc*h_img)-alto/2)
                x2 = int(x1+ancho)
                y2 = int(y1+alto)

            # --- POLYGON ---
            else:
                xs = coords[0::2]
                ys = coords[1::2]

                x1 = int(min(xs)*w_img)
                x2 = int(max(xs)*w_img)
                y1 = int(min(ys)*h_img)
                y2 = int(max(ys)*h_img)

            # ignorar mitad izquierda
            if x2 < mitad:
                continue

            # ajustar coordenadas
            x1 -= mitad
            x2 -= mitad

            x1, y1 = max(0,x1), max(0,y1)
            x2, y2 = min(mitad,x2), min(h_img,y2)

            if x2<=x1 or y2<=y1:
                continue

            roi = img[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            gris = cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)

            _,mask = cv2.threshold(gris,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

            if not np.any(mask):
                continue

            # --- COMPONENTE PRINCIPAL ---
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask,8)

            if num_labels > 1:
                largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
                mask = (labels == largest).astype(np.uint8)*255

            dist = cv2.distanceTransform(mask,cv2.DIST_L2,5)

            ske = skeletonize(mask>0)

            # --- LONGITUD ---
            length_px = calcular_longitud_skeleton(ske)
            length_cm = (length_px * px_to_mm * FACTOR_LONGITUD)/10

            # --- ANCHO ---
            if np.count_nonzero(ske)>0:
                d_vals = dist[ske>0]*2
                width_px = 0.5*np.median(d_vals) + 0.5*np.percentile(d_vals,40)
                ancho_cm = (width_px * px_to_mm * FACTOR_ANCHO)/10
            else:
                ancho_cm = 0.0

            # --- FILTRO POR RANGO REAL ---
            if not (MIN_LARGO <= length_cm <= MAX_LARGO):
                continue

            if not (MIN_ANCHO <= ancho_cm <= MAX_ANCHO):
                continue

            cv2.rectangle(canvas,(x1,y1),(x2,y2),(0,255,0),2)

            txt = f"{length_cm:.1f}x{ancho_cm:.1f}cm"

            cv2.putText(canvas, txt,(x1,max(12,y1-8)),
                        cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,255,0),2)

            hay_datos = True

        if hay_datos:
            cv2.imwrite(os.path.join(DIR_OUT,img_name),canvas)

    zip_path = '/content/resultado_final'
    shutil.make_archive(zip_path,'zip',DIR_OUT)

    try:
        files.download(zip_path+'.zip')
    except:
        print("⚠️ Descarga automática falló")

    print("✅ LISTO")


procesar_fijo_final()

🚀 Procesando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ LISTO
